# PlantSeg -- Array 2nd-Revision Notebook C: Library-Version-Drift Recovery
## (SegFormer_b2, all 3 seeds -- ViT-B/16 + Swin-Tiny, seed 42 only)

**Pure inference. No training, no new checkpoints, no hand-written key remapping.** This
notebook exists because Notebooks A and B's own diagnostics (run successfully on Kaggle)
showed two checkpoint-loading failures that are **not** file corruption and **not** a
permanent architecture incompatibility -- both are explained by the fact that Kaggle's
`!pip install -q ... transformers` / `!pip install -q timm` lines (used by every training and
evaluation notebook in this project, including the original June 2026 training) are
**unpinned**, so every run silently gets "whatever is newest on PyPI on the day it runs." The
checkpoints were trained months ago; today's Kaggle run pulled today's library versions.

### What was actually proven (not guessed) in Notebooks A/B's own printed output
1. **SegFormer_b2 (all 3 seeds -- 42, 123, 7 -- fail identically):** the `[apply_state]`
   warning shows the checkpoint's saved keys use the *old* flat `transformers` SegFormer
   layout (`segformer.encoder.patch_embeddings.{i}.*`, `segformer.encoder.block.{i}.{j}.*`,
   `segformer.encoder.layer_norm.{i}.*`, `decode_head.linear_c.{i}.*`), while the
   `transformers` version this morning's `!pip install` grabbed expects a *consolidated*
   `segformer.stages.{i}.*` layout. This is a pure **naming/structure refactor inside the
   `transformers` library**, not a corrupted file -- confirmed separately by a byte-level
   checkpoint inspection (pickle opcode disassembly) showing the `.pt` file itself is
   well-formed and exactly matches the original June 2026 file.
2. **ViT-B/16 and Swin-Tiny, seed 42 only (seed 123/seed 7 are unaffected):** a direct
   byte-comparison of the saved `data.pkl` (the part of the `.pt` zip that encodes every
   parameter's name, shape, dtype and storage layout) shows it is **100% IDENTICAL** between
   the seed-42, seed-123 and seed-7 checkpoints for both architectures -- same key names, same
   shapes, same ordering. So this is **not** a key-mismatch problem like SegFormer_b2 (which
   would show up as `[apply_state] missing=... unexpected=...` -- and it does not). The
   checkpoint loads cleanly with zero missing/unexpected keys and still produces near-chance
   accuracy. That pattern -- identical state-dict structure, silently wrong numbers -- means a
   **semantic convention inside `timm`'s ViT/Swin forward pass changed** between when the
   June-2026 seed-42 models were trained and when the September-2026 seed-123/seed-7 models
   were trained (which model code expects of a given shape, not the shape itself). The
   September checkpoints are self-consistent with *current* `timm` because they were *also
   trained* under a `timm` from on or after whatever changed; the June (seed-42) checkpoint
   was trained before that change and is now permanently "foreign" to any current `timm`
   unless evaluated under a `timm` from its own era.

### What this notebook does about it
Rather than hand-writing a state-dict key remapping (risky -- a wrong rename could silently
produce *another* wrong-but-plausible-looking number, exactly the failure mode we are trying
to eliminate), it **pins an older release** of the one library responsible, in a short
**version ladder**: try the oldest-first candidate, check whether the checkpoint now loads
with (near-)zero missing/unexpected keys, and if not, uninstall and try the next candidate --
all inside one notebook run, no kernel restart needed, because both libraries are pure-Python
and a `sys.modules` purge + clean re-import is enough to pick up a freshly `pip install`ed
version.

The version numbers in `TRANSFORMERS_LADDER` and `TIMM_LADDER` below are **best-estimate
starting guesses**, not verified facts -- this sandbox has no network access to PyPI/HF to
confirm exact release dates against this project's training dates. `TIMM_LADDER`'s first
entry (`1.0.9`) is not a guess, though -- it is the exact version already pinned and working
in `PlantSeg_Deployment_Space/requirements.txt` elsewhere in this project. If every candidate
in a ladder fails, the notebook prints that plainly, skips that recovery section without
crashing, and leaves the rest of the notebook (and whatever the other ladder already
recovered) intact -- add more/older candidates to the relevant list and re-run just this small
notebook (minutes, not hours).

## Required Kaggle inputs (attach as datasets)
Same as Notebooks A/B: the PlantSeg image+annotation dataset, the 8 original seed-42
classifier checkpoints, the 6 original seed-42 segmenter checkpoints, and the seed-123/seed-7
classifier and segmenter checkpoints from the multi-seed retraining.

## Estimated runtime
This notebook only touches 3 models (SegFormer_b2 at 3 seeds; ViT-B/16 and Swin-Tiny at seed
42 only) instead of all 14 -- expect roughly **20-40 minutes** total, even with a 4-5-candidate
version ladder for each (each failed candidate costs ~10-20s of pip install time, not a full
model pass, since the ladder checks the state-dict match *before* running any images).

## Setup

In [1]:
import os, json, glob, random, warnings, io, zipfile, subprocess, sys, importlib
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from scipy.stats import fisher_exact, mannwhitneyu
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# NOTE: timm / segmentation-models-pytorch / transformers are deliberately NOT installed yet.
# Each recovery section below installs and pins its own library version explicitly, so no
# import happens with an unpinned "whatever is newest today" version.

Device: cuda


## Config (same data-loading logic as Notebooks A/B)

In [2]:
# ============ CONFIG ============
def find_data_root():
    for cand in [Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg"),
                 Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (cand/"Metadata.csv").exists(): return cand
    hits = glob.glob("/kaggle/input/**/Metadata.csv", recursive=True)
    assert hits, "Metadata.csv not found anywhere under /kaggle/input -- attach the PlantSeg dataset."
    return Path(hits[0]).parent

DATA_ROOT = find_data_root()
print("DATA_ROOT:", DATA_ROOT)

OUTDIR = Path("/kaggle/working/r2_notebookC_results"); OUTDIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE_CLS = 384; IMG_SIZE_CLS_TF = 224; IMG_SIZE_SEG = 512
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta.columns = [c.strip() for c in meta.columns]
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
meta["maskpath"] = meta.apply(lambda r: DATA_ROOT/"annotations"/r["split"]/r["Label file"], axis=1) \
    if "Label file" in meta.columns else None
meta["label"] = meta["Disease"]
CLASS_NAMES = sorted(meta["label"].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}

train_df = meta[meta.split=="train"].reset_index(drop=True)
val_df   = meta[meta.split=="val"].reset_index(drop=True)
test_df  = meta[meta.split=="test"].reset_index(drop=True)

if meta["maskpath"].isna().any() or not meta["maskpath"].apply(lambda p: p is not None and Path(p).exists()).all():
    from collections import defaultdict
    from PIL import ImageDraw
    MASK = Path("/kaggle/working/masks")
    def load_anns(split):
        d = json.load(open(DATA_ROOT/f"annotation_{split}.json")); by = defaultdict(list)
        for a in d["annotations"]: by[a["image_id"]].append(a)
        fn = {}
        for im in d["images"]:
            polys = [seg for a in by.get(im["id"], []) for seg in a["segmentation"] if len(seg) >= 6]
            fn[im["file_name"]] = (im["width"], im["height"], polys)
        return fn
    def rasterize(w, h, polys):
        m = Image.new("L", (w, h), 0); dr = ImageDraw.Draw(m)
        for seg in polys: dr.polygon(list(zip(seg[0::2], seg[1::2])), fill=1)
        return m
    mp_by_name = {}
    for split in ["train", "val", "test"]:
        fn = load_anns(split); od = MASK/split; od.mkdir(parents=True, exist_ok=True)
        for r in meta[meta.split == split].itertuples():
            w, h, polys = fn.get(r.Name, (None, None, []))
            mp = od/(Path(r.Name).stem + ".png")
            if not mp.exists():
                if w is None:
                    with Image.open(r.filepath) as i0: w, h = i0.size
                rasterize(w, h, polys).save(mp)
            mp_by_name[r.Name] = mp
    meta["maskpath"] = meta["Name"].map(mp_by_name)
    train_df = meta[meta.split=="train"].reset_index(drop=True)
    val_df   = meta[meta.split=="val"].reset_index(drop=True)
    test_df  = meta[meta.split=="test"].reset_index(drop=True)
    print("Rasterized masks from annotation_*.json.")

print(f"classes={NUM_CLASSES}  train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

SEEDS = [42, 123, 7]
def cls_ckpt_candidates(name, seed):
    return [f"{name}_best.pt"] if seed == 42 else [f"{name}_seed{seed}_best.pt"]
def seg_ckpt_candidates(name, seed):
    return [f"{name}.pt"] if seed == 42 else [f"{name}_seed{seed}.pt", f"{name}_seed{seed}.zip"]
print("Config ready.")

DATA_ROOT: /kaggle/input/datasets/alenmaruf/plantseg/plantseg
classes=115  train=5367  val=846  test=1561
Config ready.


## Robust checkpoint finder (identical to Notebook B)

In [3]:
# ============ ROBUST CHECKPOINT FINDER ============
def _rezip_dir_as_pt(dirpath: Path):
    root_name = dirpath.name
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_STORED) as zf:
        for f in dirpath.rglob("*"):
            if f.is_file():
                zf.write(f, arcname=f"{root_name}/{f.relative_to(dirpath).as_posix()}")
    buf.seek(0)
    return buf

_ckpt_cache = {}

def find_and_load_checkpoint(filename_candidates, search_root="/kaggle/input"):
    key = tuple(filename_candidates)
    if key in _ckpt_cache:
        return _ckpt_cache[key]
    for fname in filename_candidates:
        hits = glob.glob(f"{search_root}/**/{fname}", recursive=True)
        for h in hits:
            hp = Path(h)
            try:
                state = torch.load(hp, map_location="cpu")
                print(f"    [find_and_load] loaded {hp}")
                _ckpt_cache[key] = state; return state
            except Exception as e:
                print(f"    [find_and_load] {hp} exists but torch.load failed directly ({type(e).__name__}: {e});"
                      f" checking for an auto-extracted folder instead...")
        stem = Path(fname).stem
        dir_hits = [d for d in glob.glob(f"{search_root}/**/{stem}", recursive=True) if Path(d).is_dir()]
        for d in dir_hits:
            dp = Path(d)
            if (dp / "data.pkl").exists():
                print(f"    [find_and_load] found extracted checkpoint folder {dp}; re-zipping in memory")
                state = torch.load(_rezip_dir_as_pt(dp), map_location="cpu")
                _ckpt_cache[key] = state; return state
            inner = dp / stem
            if inner.is_dir() and (inner / "data.pkl").exists():
                print(f"    [find_and_load] found nested extracted checkpoint folder {inner}; re-zipping in memory")
                state = torch.load(_rezip_dir_as_pt(inner), map_location="cpu")
                _ckpt_cache[key] = state; return state
    raise FileNotFoundError(f"None of {filename_candidates} found (as file or extracted folder) under {search_root}")

def _unwrap_state(raw):
    if isinstance(raw, dict):
        for key in ("state_dict", "model_state_dict", "model", "best_state"):
            if key in raw and isinstance(raw[key], dict):
                raw = raw[key]; break
    return {k.replace("module.", ""): v for k, v in raw.items()}

def soft_apply_state(model, raw):
    # Like apply_state, but NEVER raises -- returns (ok, missing_count, unexpected_count) so a
    # version-ladder loop can try the next candidate instead of crashing the notebook.
    raw = _unwrap_state(raw)
    missing, unexpected = model.load_state_dict(raw, strict=False)
    total_model_keys = len(model.state_dict())
    severe = total_model_keys and len(missing) >= total_model_keys * 0.5
    ok = not severe
    return ok, len(missing), len(unexpected)

def apply_state(model, raw):
    # Strict, raising version -- used once a ladder has already found a clean-matching
    # library version, for the final real metric computation.
    raw = _unwrap_state(raw)
    missing, unexpected = model.load_state_dict(raw, strict=False)
    if missing or unexpected:
        print(f"    [apply_state] missing={len(missing)} unexpected={len(unexpected)}")
        total_model_keys = len(model.state_dict())
        if total_model_keys and len(missing) >= total_model_keys * 0.5:
            print(f"    [apply_state] sample checkpoint keys: {list(raw.keys())[:3]}")
            print(f"    [apply_state] sample expected keys  : {list(model.state_dict().keys())[:3]}")
            raise RuntimeError(f">=50% of keys unmatched (missing={len(missing)}, unexpected={len(unexpected)}) "
                                f"-- checkpoint did not load; refusing to report numbers from random weights")
    return model

print("Checkpoint finder ready.")

Checkpoint finder ready.


## 300-image stratified subset (identical list to Notebooks A/B / Stage 4/6)

In [4]:
NAMES_300 = ['apple_black_rot_google_0037.jpg', 'apple_black_rot_google_0077.jpg', 'apple_mosaic_virus_57.jpg', 'apple_mosaic_virus_google_0074.jpg', 'apple_rust_147.jpg', 'apple_rust_google_0007.jpg', 'apple_rust_google_0011.jpg', 'apple_scab_50.jpg', 'apple_scab_85.jpg', 'apple_scab_google_0018.jpg', 'apple_scab_google_0031.jpg', 'apple_scab_google_0153.jpg', 'apple_scab_google_0244.jpg', 'banana_anthracnose_Baidu_0059.jpg', 'banana_anthracnose_Bing_0006.jpg', 'banana_anthracnose_Bing_0022.jpg', 'banana_anthracnose_Bing_0032.jpg', 'banana_black_leaf_streak_Bing_0124.jpg', 'banana_black_leaf_streak_banana black sigatoka (26).jpg', 'banana_bunchy_top_Bing_0152.jpg', 'banana_bunchy_top_Google_0037.jpg', 'banana_cigar_end_rot_Bing_0007.jpg', 'banana_cigar_end_rot_Google_0101.jpg', 'banana_cordana_leaf_spot_Bing_0007.jpg', 'banana_cordana_leaf_spot_Bing_0037.jpg', 'banana_panama_disease_126.jpg', 'banana_panama_disease_google_0064.jpg', 'basil_downy_mildew_google_0031.jpg', 'basil_downy_mildew_google_0091.jpg', 'bean_halo_blight_11.jpg', 'bean_halo_blight_2.jpg', 'bean_mosaic_virus_24.jpg', 'bean_mosaic_virus_google_0098.jpg', 'bean_rust_google_0045.jpg', 'bean_rust_google_0068.jpg', 'bell_pepper_bacterial_spot_Bing_0002.jpg', 'bell_pepper_bacterial_spot_Bing_0028.jpg', 'bell_pepper_blossom_end_rot_Bing_0002.jpg', 'bell_pepper_blossom_end_rot_Bing_0004.jpg', 'bell_pepper_blossom_end_rot_Bing_0077.jpg', 'bell_pepper_frogeye_leaf_spot_Bing_0008.jpg', 'bell_pepper_frogeye_leaf_spot_Google_0049.jpg', 'bell_pepper_powdery_mildew_Bing_0035.jpg', 'bell_pepper_powdery_mildew_Google_0118.jpg', 'blueberry_anthracnose_Bing_0006.jpg', 'blueberry_anthracnose_Bing_0145.jpg', 'blueberry_botrytis_blight_Bing_0015.jpg', 'blueberry_botrytis_blight_Bing_0021.jpg', 'blueberry_mummy_berry_Bing_0024.jpg', 'blueberry_mummy_berry_Google_0038.jpg', 'blueberry_rust_google_0004.jpg', 'blueberry_rust_google_0053.jpg', 'blueberry_scorch_Bing_0011.jpg', 'blueberry_scorch_Bing_0031.jpg', 'blueberry_scorch_Bing_0062.jpg', 'broccoli_alternaria_leaf_spot_Bing_0000.jpg', 'broccoli_alternaria_leaf_spot_Bing_0054.jpg', 'broccoli_downy_mildew_30.jpg', 'broccoli_downy_mildew_40.jpg', 'broccoli_ring_spot_Bing_0002.jpg', 'broccoli_ring_spot_Bing_0018.jpg', 'cabbage_alternaria_leaf_spot_3.jpg', 'cabbage_alternaria_leaf_spot_google_0206.jpg', 'cabbage_black_rot_Bing_0002.jpg', 'cabbage_black_rot_Bing_0018.jpg', 'cabbage_black_rot_Bing_0039.jpg', 'cabbage_black_rot_Bing_0080.jpg', 'cabbage_black_rot_Bing_0114.jpg', 'cabbage_downy_mildew_Bing_0012.jpg', 'cabbage_downy_mildew_Bing_0124.jpg', 'carrot_alternaria_leaf_blight_Bing_0046.jpg', 'carrot_alternaria_leaf_blight_Bing_0071.jpg', 'carrot_cavity_spot_google_0000.jpg', 'carrot_cavity_spot_google_0161.jpg', 'carrot_cercospora_leaf_blight_Bing_0009.jpg', 'carrot_cercospora_leaf_blight_Bing_0212.jpg', 'cauliflower_alternaria_leaf_spot_google_0003.jpg', 'cauliflower_alternaria_leaf_spot_google_0061.jpg', 'cauliflower_bacterial_soft_rot_Bing_0021.jpg', 'cauliflower_bacterial_soft_rot_Bing_0205.jpg', 'celery_anthracnose_google_0007.jpg', 'celery_anthracnose_google_0035.jpg', 'celery_early_blight_52.jpg', 'celery_early_blight_google_0001.jpg', 'celery_early_blight_google_0009.jpg', 'cherry_leaf_spot_google_0180.jpg', 'cherry_leaf_spot_google_0216.jpg', 'cherry_powdery_mildew_107.jpg', 'cherry_powdery_mildew_2.jpg', 'citrus_canker_13.jpg', 'citrus_canker_147.jpg', 'citrus_canker_163.jpg', 'citrus_canker_247.jpg', 'citrus_canker_363.jpg', 'citrus_canker_76.jpg', 'citrus_canker_89.jpg', 'citrus_canker_google_0071.jpg', 'citrus_greening_disease_29.jpg', 'citrus_greening_disease_48.jpg', 'citrus_greening_disease_google_0120.jpg', 'citrus_greening_disease_google_0251.jpg', 'coffee_berry_blotch_Bing_0027.jpg', 'coffee_berry_blotch_Bing_0114.jpg', 'coffee_berry_blotch_Bing_0120.jpg', 'coffee_brown_eye_spot_Bing_0036.jpg', 'coffee_brown_eye_spot_Google_0003.jpg', 'coffee_leaf_rust_15.jpg', 'coffee_leaf_rust_google_0016.jpg', 'coffee_leaf_rust_google_0135.jpg', 'corn_gray_leaf_spot_31.jpg', 'corn_gray_leaf_spot_35.jpg', 'corn_gray_leaf_spot_google_0074.jpg', 'corn_northern_leaf_blight_101.jpg', 'corn_northern_leaf_blight_14.jpg', 'corn_northern_leaf_blight_81.jpg', 'corn_rust_80.jpg', 'corn_rust_google_0078.jpg', 'corn_smut_google_0013.jpg', 'corn_smut_google_0063.jpg', 'corn_smut_google_0079.jpg', 'cucumber_angular_leaf_spot_1.jpg', 'cucumber_angular_leaf_spot_156.jpg', 'cucumber_angular_leaf_spot_328.jpg', 'cucumber_angular_leaf_spot_41.jpg', 'cucumber_angular_leaf_spot_google_0043.jpg', 'cucumber_bacterial_wilt_109.jpg', 'cucumber_bacterial_wilt_313.jpg', 'cucumber_bacterial_wilt_50.jpg', 'cucumber_powdery_mildew_58.jpg', 'cucumber_powdery_mildew_70.jpg', 'cucumber_powdery_mildew_77.jpg', 'cucumber_powdery_mildew_google_0159.jpg', 'eggplant_cercospora_leaf_spot_12.jpg', 'eggplant_cercospora_leaf_spot_33.jpg', 'eggplant_cercospora_leaf_spot_40.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0016.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0022.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0261.jpg', 'eggplant_phytophthora_blight_Bing_0000.jpg', 'eggplant_phytophthora_blight_Bing_0002.jpg', 'garlic_leaf_blight_140.jpg', 'garlic_leaf_blight_46.jpg', 'garlic_leaf_blight_google_0004.jpg', 'garlic_leaf_blight_google_0210.jpg', 'garlic_rust_159.jpg', 'garlic_rust_18.jpg', 'ginger_leaf_spot_11.jpg', 'ginger_leaf_spot_google_0001.jpg', 'ginger_sheath_blight_53.jpg', 'ginger_sheath_blight_89.jpg', 'grape_black_rot_12.jpg', 'grape_black_rot_216.jpg', 'grape_black_rot_google_0013.jpg', 'grape_downy_mildew_251.jpg', 'grape_downy_mildew_272.jpg', 'grape_downy_mildew_google_0032.jpg', 'grape_downy_mildew_google_0052.jpg', 'grape_leaf_spot_101.jpg', 'grape_leaf_spot_168.jpg', 'grapevine_leafroll_disease_google_0001.jpg', 'grapevine_leafroll_disease_google_0181.jpg', 'lettuce_downy_mildew_google_0015.jpg', 'lettuce_downy_mildew_google_0027.jpg', 'lettuce_mosaic_virus_16.jpg', 'lettuce_mosaic_virus_24.jpg', 'maple_tar_spot_google_0005.jpg', 'maple_tar_spot_google_0089.jpg', 'peach_anthracnose_Bing_0002.jpg', 'peach_anthracnose_Bing_0013.jpg', 'peach_brown_rot_Bing_0000.jpg', 'peach_brown_rot_Bing_0092.jpg', 'peach_brown_rot_Bing_0114.jpg', 'peach_leaf_curl_34.jpg', 'peach_leaf_curl_google_0020.jpg', 'peach_leaf_curl_google_0035.jpg', 'peach_leaf_curl_google_0074.jpg', 'peach_rust_Bing_0001.jpg', 'peach_scab_Bing_0007.jpg', 'peach_scab_Bing_0083.jpg', 'plum_bacterial_spot_Bing_0013.jpg', 'plum_bacterial_spot_Bing_0082.jpg', 'plum_brown_rot_Bing_0001.jpg', 'plum_brown_rot_Bing_0097.jpg', 'plum_pocket_disease_google_0028.jpg', 'plum_pocket_disease_google_0032.jpg', 'plum_pocket_disease_google_0117.jpg', 'plum_pox_virus_Bing_0052.jpg', 'plum_pox_virus_Bing_0054.jpg', 'plum_pox_virus_Google_0040.jpg', 'plum_rust_Bing_0024.jpg', 'plum_rust_Google_0036.jpg', 'potato_early_blight_6.jpg', 'potato_early_blight_google_0095.jpg', 'potato_late_blight_11.jpg', 'potato_late_blight_36.jpg', 'potato_late_blight_96.jpg', 'raspberry_fire_blight_Bing_0018.jpg', 'raspberry_fire_blight_Google_0005.jpg', 'raspberry_gray_mold_Bing_0000.jpg', 'raspberry_gray_mold_Bing_0001.jpg', 'raspberry_leaf_spot_Bing_0073.jpg', 'raspberry_yellow_rust_Bing_0010.jpg', 'raspberry_yellow_rust_Bing_0183.jpg', 'rice_blast_151.jpg', 'rice_blast_87.jpg', 'rice_sheath_blight_148.jpg', 'rice_sheath_blight_google_0027.jpg', 'soybean_bacterial_blight_Bing_0057.jpg', 'soybean_bacterial_blight_Bing_0105.jpg', 'soybean_bacterial_blight_Bing_0172.jpg', 'soybean_brown_spot_Baidu_0010.jpg', 'soybean_brown_spot_Bing_0353.jpg', 'soybean_downy_mildew_Bing_0078.jpg', 'soybean_downy_mildew_Bing_0143.jpg', 'soybean_frog_eye_leaf_spot_Bing_0141.jpg', 'soybean_frog_eye_leaf_spot_Google_0057.jpg', 'soybean_frog_eye_leaf_spot_Google_0221.jpg', 'soybean_mosaic_Baidu_0059.jpg', 'soybean_mosaic_Baidu_0089.jpg', 'soybean_rust_Bing_0022.jpg', 'soybean_rust_Bing_0055.jpg', 'soybean_rust_Bing_0074.jpg', 'soybean_rust_Bing_0075.jpg', 'soybean_rust_Bing_0096.jpg', 'squash_powdery_mildew_129.jpg', 'squash_powdery_mildew_23.jpg', 'squash_powdery_mildew_6.jpg', 'squash_powdery_mildew_google_0105.jpg', 'strawberry_anthracnose_5.jpg', 'strawberry_anthracnose_google_0001.jpg', 'strawberry_leaf_scorch_google_0037.jpg', 'strawberry_leaf_scorch_google_0133.jpg', 'tobacco_blue_mold_Bing_0024.jpg', 'tobacco_blue_mold_Bing_0096.jpg', 'tobacco_brown_spot_Bing_0001.jpg', 'tobacco_brown_spot_Bing_0047.jpg', 'tobacco_brown_spot_Bing_0055.jpg', 'tobacco_frogeye_leaf_spot_Bing_0002.jpg', 'tobacco_frogeye_leaf_spot_Google_0052.jpg', 'tobacco_mosaic_virus_117.jpg', 'tobacco_mosaic_virus_4.jpg', 'tomato_bacterial_leaf_spot_120.jpg', 'tomato_bacterial_leaf_spot_14.jpg', 'tomato_early_blight_96.jpg', 'tomato_early_blight_google_0059.jpg', 'tomato_early_blight_google_0177.jpg', 'tomato_late_blight_10.jpg', 'tomato_late_blight_182.jpg', 'tomato_late_blight_205.jpg', 'tomato_leaf_mold_10.jpg', 'tomato_leaf_mold_148.jpg', 'tomato_mosaic_virus_15.jpg', 'tomato_mosaic_virus_google_0049.jpg', 'tomato_septoria_leaf_spot_10.jpg', 'tomato_septoria_leaf_spot_18.jpg', 'tomato_septoria_leaf_spot_243.jpg', 'tomato_septoria_leaf_spot_80.jpg', 'tomato_septoria_leaf_spot_google_0027.jpg', 'tomato_septoria_leaf_spot_google_0070.jpg', 'tomato_septoria_leaf_spot_google_0128.jpg', 'tomato_yellow_leaf_curl_virus_15.jpg', 'tomato_yellow_leaf_curl_virus_17.jpg', 'tomato_yellow_leaf_curl_virus_40.jpg', 'tomato_yellow_leaf_curl_virus_99.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0031.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0235.jpg', 'wheat_head_scab_Baidu_0481.jpg', 'wheat_head_scab_Bing_0040.jpg', 'wheat_head_scab_Bing_0043.jpg', 'wheat_head_scab_Bing_0288.jpg', 'wheat_leaf_rust_Google_0040.jpg', 'wheat_leaf_rust_Google_0058.jpg', 'wheat_leaf_rust_Google_0328.jpg', 'wheat_loose_smut_Bing_0109.jpg', 'wheat_loose_smut_Bing_0283.jpg', 'wheat_loose_smut_Google_0081.jpg', 'wheat_powdery_mildew_Baidu_0181.jpg', 'wheat_powdery_mildew_Bing_0014.jpg', 'wheat_powdery_mildew_Google_0014.jpg', 'wheat_septoria_blotch_Google_0115.jpg', 'wheat_septoria_blotch_google_septoria (18).jpg', 'wheat_stem_rust_Baidu_0001.jpg', 'wheat_stem_rust_Bing_0217.jpg', 'wheat_stem_rust_Google_0274.jpg', 'wheat_stripe_rust_Baidu_0143.jpg', 'wheat_stripe_rust_Bing_0007.jpg', 'wheat_stripe_rust_Bing_0257.jpg', 'wheat_stripe_rust_Google_0029.jpg', 'zucchini_bacterial_wilt_Bing_0105.jpg', 'zucchini_bacterial_wilt_Bing_0188.jpg', 'zucchini_bacterial_wilt_Bing_0260.jpg', 'zucchini_bacterial_wilt_Google_0060.jpg', 'zucchini_downy_mildew_Bing_0035.jpg', 'zucchini_downy_mildew_Bing_0259.jpg', 'zucchini_powdery_mildew_Bing_0109.jpg', 'zucchini_powdery_mildew_Bing_0155.jpg', 'zucchini_powdery_mildew_Bing_0233.jpg', 'zucchini_yellow_mosaic_virus_11.jpg', 'zucchini_yellow_mosaic_virus_google_0038.jpg', 'zucchini_yellow_mosaic_virus_google_0067.jpg']
print(f"NAMES_300 loaded: {len(NAMES_300)} images")

NAMES_300 loaded: 300 images


## Version-ladder installer (pure-Python packages only -- no kernel restart needed)

In [5]:
# ============ VERSION LADDER UTILITY ============
def pip_install_version(pkg, version):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{pkg}=={version}"],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)

def purge_module(prefix):
    for mod in list(sys.modules):
        if mod == prefix or mod.startswith(prefix + "."):
            del sys.modules[mod]

def try_version_ladder(pkg, ladder, probe_fn):
    # probe_fn() must: fresh-import the package, build the model, attempt soft_apply_state,
    # and return (ok: bool, info: dict) -- info is logged either way, and on success is kept
    # for the real computation that follows. Tries `ladder` oldest-guess-first; stops at the
    # first version that reports ok=True.
    for v in ladder:
        print(f"  -- trying {pkg}=={v} --")
        try:
            pip_install_version(pkg, v)
        except subprocess.CalledProcessError as e:
            print(f"     [{pkg}=={v}] pip install failed (version may not exist on PyPI): "
                  f"{e.stderr.decode()[-300:] if e.stderr else e}")
            continue
        purge_module(pkg)
        try:
            ok, info = probe_fn()
        except Exception as e:
            print(f"     [{pkg}=={v}] probe raised: {type(e).__name__}: {e}")
            continue
        if ok:
            print(f"  >>> SUCCESS: {pkg}=={v} loads this checkpoint cleanly. {info}")
            return v, info
        else:
            print(f"     [{pkg}=={v}] still mismatched: {info}")
    print(f"  *** No candidate in {ladder} worked for {pkg}. Add older versions to the ladder and re-run "
          f"just this notebook (it is short) -- this recovery section will be skipped for now. ***")
    return None, None

print("Version-ladder utility ready.")

Version-ladder utility ready.


## Section 1 -- SegFormer_b2 recovery (all 3 seeds)

Pins an older `transformers` release so the checkpoint's saved flat
`encoder.patch_embeddings/block/layer_norm` key layout matches what the installed library
expects, instead of the newer consolidated `stages.{i}.*` layout. All 3 seeds are tried with
whichever version first succeeds on seed 42 (the per-image Kaggle log already shows all 3
seeds fail identically, so one working version should recover all 3 -- the loop still
verifies each seed independently rather than assuming this).

In [6]:
# ============ SECTION 1: SEGFORMER_B2 RECOVERY ============
# Best-estimate starting guesses only (this sandbox cannot query PyPI to confirm exact
# release dates against this project's training dates) -- oldest-guess-first. Add more/older
# entries here and re-run this notebook if none of these land a clean match.
TRANSFORMERS_LADDER = ["4.46.3", "4.44.2", "4.41.2", "4.38.2", "4.35.2"]

SEGFORMER_HF_ID = "nvidia/mit-b2"
_sf_working_version = None

def _build_segformer_shell():
    from transformers import SegformerForSemanticSegmentation
    class _SegNetSF(nn.Module):
        def __init__(self, nc):
            super().__init__()
            self.net = SegformerForSemanticSegmentation.from_pretrained(
                SEGFORMER_HF_ID, num_labels=nc, ignore_mismatched_sizes=True)
        def forward(self, x):
            lg = self.net(pixel_values=x).logits
            return F.interpolate(lg, size=x.shape[-2:], mode="bilinear", align_corners=False)
    return _SegNetSF(2)

def _probe_segformer(seed42_raw):
    model = _build_segformer_shell()
    ok, n_missing, n_unexpected = soft_apply_state(model, seed42_raw)
    return ok, {"missing": n_missing, "unexpected": n_unexpected}

try:
    seed42_raw = find_and_load_checkpoint(seg_ckpt_candidates("SegFormer_b2", 42))
    _sf_working_version, _sf_info = try_version_ladder(
        "transformers", TRANSFORMERS_LADDER, lambda: _probe_segformer(seed42_raw))
except FileNotFoundError as e:
    print(f"  [skip] SegFormer_b2 seed42 checkpoint not found: {e}")

sf_val_rows = []
sf_recompute_rows = []
sf_chance_rows = []

if _sf_working_version is not None:
    from transformers import SegformerForSemanticSegmentation

    class SegNetSF(nn.Module):
        def __init__(self, nc):
            super().__init__()
            self.net = SegformerForSemanticSegmentation.from_pretrained(
                SEGFORMER_HF_ID, num_labels=nc, ignore_mismatched_sizes=True); self.kind = "hf"
        def forward(self, x):
            lg = self.net(pixel_values=x).logits
            return F.interpolate(lg, size=x.shape[-2:], mode="bilinear", align_corners=False)

    seg_eval_tf = T.Compose([T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG)), T.ToTensor(), T.Normalize(MEAN, STD)])
    mask_tf = T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG), interpolation=T.InterpolationMode.NEAREST)

    def evaluate_segmenter_on(df, model):
        model.eval(); ious = []; dices = []
        with torch.no_grad():
            for r in df.itertuples():
                x = seg_eval_tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
                gt = np.array(mask_tf(Image.open(r.maskpath))) > 0
                pr = (model(x).argmax(1)[0].cpu().numpy() > 0)
                inter = (gt & pr).sum(); union = (gt | pr).sum()
                ious.append(inter/union if union > 0 else 1.0)
                dices.append(2*inter/(gt.sum()+pr.sum()) if (gt.sum()+pr.sum()) > 0 else 1.0)
        return float(np.mean(ious)), float(np.mean(dices))

    def ccc_fn(y_true, y_pred):
        y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
        mt, mp = y_true.mean(), y_pred.mean(); vt, vp = y_true.var(), y_pred.var()
        covar = np.mean((y_true-mt)*(y_pred-mp))
        return (2*covar)/(vt+vp+(mt-mp)**2)

    class _GradCAMHookSF:
        pass  # SegFormer uses attention-rollout, not Grad-CAM -- see segformer_attention_map below

    def segformer_attention_map(model, x):
        with torch.no_grad():
            out = model.net.segformer(pixel_values=x, output_attentions=True)
        attns = out.attentions
        if not attns:
            raise RuntimeError("segformer_attention_map: model returned no attentions -- "
                                "version ladder reported success but forward pass still looks wrong")
        stages=[]; cur=[]; cur_n=None
        for a in attns:
            n=a.shape[-2]
            if cur_n is None or n==cur_n: cur.append(a); cur_n=n
            else: stages.append(cur); cur=[a]; cur_n=n
        stages.append(cur)
        rollout=None
        for a in stages[-1]:
            A = a[0].mean(0); A = A/A.sum(-1,keepdim=True).clamp_min(1e-8)
            A_hat = 0.5*A + 0.5*torch.eye(A.shape[0], device=A.device)
            rollout = A_hat if rollout is None else A_hat@rollout
        imp = rollout.mean(0); side = int(round(imp.shape[0]**0.5))
        grid = imp[:side*side].reshape(side,side).float(); grid = grid/(grid.max()+1e-8)
        cam  = F.interpolate(grid[None,None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0,0]
        return cam.detach().cpu().numpy()

    def cam_gt_metrics(cam, gt_mask):
        if gt_mask.sum() == 0: return {"iou": np.nan, "pointing_hit": np.nan}
        thresh = np.quantile(cam, 1 - gt_mask.mean())
        cam_bin = cam >= thresh
        inter = (cam_bin & gt_mask).sum(); union = (cam_bin | gt_mask).sum()
        iou = float(inter/union) if union > 0 else np.nan
        py, px = np.unravel_index(np.argmax(cam), cam.shape)
        return {"iou": iou, "pointing_hit": bool(gt_mask[py, px])}

    def deletion_auc(model, x, cam, steps=10):
        H, W = x.shape[-2:]; order = np.argsort(-cam.ravel()); npix = order.size
        baseline = x.mean(dim=(2,3), keepdim=True); fracs = np.linspace(0, 1, steps+1); probs = []
        with torch.no_grad():
            for f in fracs:
                k = int(f*npix); xin = x.clone()
                if k > 0:
                    mask = torch.zeros(npix, dtype=torch.bool, device=x.device)
                    mask[order[:k]] = True; mask = mask.reshape(H, W)
                    for c in range(xin.shape[1]): xin[0,c][mask] = baseline[0,c,0,0]
                logits = model(xin)
                p = torch.softmax(logits,1)[0,1:].sum(0).mean().item()
                probs.append(p)
        return float(np.trapz(probs, fracs))

    # ---- 1a. Validation-split re-ranking + full test-set recompute, 3 seeds ----
    for seed in SEEDS:
        try:
            raw = find_and_load_checkpoint(seg_ckpt_candidates("SegFormer_b2", seed))
        except FileNotFoundError as e:
            print(f"  [skip] seed{seed} SegFormer_b2: {e}"); continue
        try:
            model = SegNetSF(2); model = apply_state(model, raw).to(device).eval()
            val_fg_iou, val_dice = evaluate_segmenter_on(val_df, model)
            sf_val_rows.append({"seed": seed, "model": "SegFormer_b2",
                                 "val_fg_iou_macro": val_fg_iou, "val_dice_macro": val_dice})
            print(f"  seed{seed} SegFormer_b2: val_fg_iou={val_fg_iou:.4f} val_dice={val_dice:.4f}")

            test_fg_iou, test_dice = evaluate_segmenter_on(test_df, model)
            sev_pred=[]; sev_gt=[]
            with torch.no_grad():
                for _, r in test_df.iterrows():
                    if "Mask ratio" not in r or pd.isna(r["Mask ratio"]): continue
                    x = seg_eval_tf(Image.open(r["filepath"]).convert("RGB")).unsqueeze(0).to(device)
                    pr = (model(x).argmax(1)[0].cpu().numpy() > 0)
                    sev_pred.append(float(pr.mean())); sev_gt.append(float(r["Mask ratio"]))
            severity_ccc = ccc_fn(sev_gt, sev_pred) if len(sev_gt) > 1 else np.nan
            severity_mae = float(np.mean(np.abs(np.array(sev_pred)-np.array(sev_gt)))) if sev_gt else np.nan

            sub_df = test_df[test_df["Name"].isin(NAMES_300)].reset_index(drop=True)
            ious=[]; hits=[]; delaucs=[]
            for r in sub_df.itertuples():
                x = seg_eval_tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
                gt = np.array(mask_tf(Image.open(r.maskpath))) > 0
                cam = segformer_attention_map(model, x)
                m = cam_gt_metrics(cam, gt)
                ious.append(m["iou"]); hits.append(m["pointing_hit"])
                delaucs.append(deletion_auc(model, x, cam))
                if seed == 42:
                    sf_chance_rows.append({"model": "SegFormer_b2", "image": r.Name, "real_iou": m["iou"],
                                            "real_pointing_hit": m["pointing_hit"], "real_deletion_auc": delaucs[-1]})

            sf_recompute_rows.append({"seed": seed, "model": "SegFormer_b2",
                                       "test_fg_iou": test_fg_iou, "test_dice": test_dice,
                                       "severity_ccc": severity_ccc, "severity_mae": severity_mae,
                                       "cam_gt_iou_mean": float(np.nanmean(ious)),
                                       "pointing_hit_mean": float(np.nanmean(hits)),
                                       "deletion_auc_mean": float(np.nanmean(delaucs))})
            print(f"  seed{seed} SegFormer_b2: test_fg_iou={test_fg_iou:.4f} severity_ccc={severity_ccc:.4f} "
                  f"cam_gt_iou={np.nanmean(ious):.4f}")
        except Exception as e:
            print(f"  [SKIP] seed{seed} SegFormer_b2: failed even with transformers=={_sf_working_version} -- {e}")
        finally:
            try: del model
            except NameError: pass
            torch.cuda.empty_cache()

pd.DataFrame(sf_val_rows).to_csv(OUTDIR/"segformer_b2_val_reranking_3seed.csv", index=False)
pd.DataFrame(sf_recompute_rows).to_csv(OUTDIR/"segformer_b2_3seed_full_recompute.csv", index=False)
pd.DataFrame(sf_chance_rows).to_csv(OUTDIR/"segformer_b2_chance_baseline_per_image_seed42.csv", index=False)
print(f"\nWorking transformers version for SegFormer_b2: {_sf_working_version}")
print("Saved: segformer_b2_val_reranking_3seed.csv, segformer_b2_3seed_full_recompute.csv, "
      "segformer_b2_chance_baseline_per_image_seed42.csv")

    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/Segmentation/43seed_Seegmentation_checkpoints/SegFormer_b2.pt
  -- trying transformers==4.46.3 --


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b2 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  >>> SUCCESS: transformers==4.46.3 loads this checkpoint cleanly. {'missing': 0, 'unexpected': 0}


...

  seed42 SegFormer_b2: val_fg_iou=0.5265 val_dice=0.6435
  seed42 SegFormer_b2: test_fg_iou=0.5327 severity_ccc=0.7608 cam_gt_iou=0.0687
    [find_and_load] found nested extracted checkpoint folder /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/Segmentation/123seed_Seegmentation_checkpoints/SegFormer_b2_seed123/SegFormer_b2_seed123; re-zipping in memory


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b2 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  seed123 SegFormer_b2: val_fg_iou=0.5661 val_dice=0.6786
  seed123 SegFormer_b2: test_fg_iou=0.5764 severity_ccc=0.8078 cam_gt_iou=0.2490
    [find_and_load] found nested extracted checkpoint folder /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/Segmentation/7seed_Seegmentation_checkpoints/SegFormer_b2_seed7/SegFormer_b2_seed7; re-zipping in memory


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b2 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  seed7 SegFormer_b2: val_fg_iou=0.4990 val_dice=0.6161
  seed7 SegFormer_b2: test_fg_iou=0.5054 severity_ccc=0.7431 cam_gt_iou=0.1857

Working transformers version for SegFormer_b2: 4.46.3
Saved: segformer_b2_val_reranking_3seed.csv, segformer_b2_3seed_full_recompute.csv, segformer_b2_chance_baseline_per_image_seed42.csv


## Section 2 -- ViT-B/16 + Swin-Tiny seed-42 recovery

Pins an older `timm` release for these two architectures' seed-42 checkpoints *only* --
seed-123/seed-7 already evaluate correctly under the current/default `timm` (confirmed in
Notebook B) and are intentionally left alone. The first candidate is `1.0.9`, the version
already pinned and working for this exact project in
`PlantSeg_Deployment_Space/requirements.txt` -- not a guess.

In [7]:
# ============ SECTION 2: VIT-B/16 + SWIN-TINY SEED-42 RECOVERY ============
TIMM_LADDER = ["1.0.9", "0.9.16", "0.9.12", "0.9.7", "0.9.2"]

TF_MODEL_IDS = {"ViTBase16": "vit_base_patch16_224", "SwinTiny": "swin_tiny_patch4_window7_224"}
_timm_working_version = None

def _probe_timm(name, raw):
    import timm as _timm
    model = _timm.create_model(TF_MODEL_IDS[name], pretrained=False, num_classes=NUM_CLASSES)
    ok, n_missing, n_unexpected = soft_apply_state(model, raw)
    return ok, {"missing": n_missing, "unexpected": n_unexpected}

try:
    vit_raw = find_and_load_checkpoint(cls_ckpt_candidates("ViTBase16", 42))
    _timm_working_version, _timm_info = try_version_ladder(
        "timm", TIMM_LADDER, lambda: _probe_timm("ViTBase16", vit_raw))
    # Sanity-check the SAME candidate also cleanly loads Swin-Tiny before committing to it --
    # if it doesn't, fall through to trying the remaining ladder entries for Swin specifically.
    if _timm_working_version is not None:
        swin_raw_check = find_and_load_checkpoint(cls_ckpt_candidates("SwinTiny", 42))
        ok_swin, info_swin = _probe_timm("SwinTiny", swin_raw_check)
        if not ok_swin:
            print(f"  NOTE: timm=={_timm_working_version} fixed ViTBase16 but not SwinTiny ({info_swin}); "
                  f"trying the remaining ladder entries for SwinTiny only.")
            remaining = TIMM_LADDER[TIMM_LADDER.index(_timm_working_version)+1:]
            _timm_working_version_swin, _ = try_version_ladder(
                "timm", remaining, lambda: _probe_timm("SwinTiny", swin_raw_check))
        else:
            _timm_working_version_swin = _timm_working_version
    else:
        _timm_working_version_swin = None
except FileNotFoundError as e:
    print(f"  [skip] ViTBase16/SwinTiny seed42 checkpoint not found: {e}")
    _timm_working_version_swin = None

tf_val_rows = []
tf_tta_rows = []

def build_backbone_tf(name, num_classes, timm_mod):
    if name == "ViTBase16":
        return timm_mod.create_model("vit_base_patch16_224", pretrained=False, num_classes=num_classes)
    if name == "SwinTiny":
        return timm_mod.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=num_classes)
    raise ValueError(name)

def evaluate_classifier_on(df, model, img_size):
    tf = T.Compose([T.Resize((img_size, img_size)), T.ToTensor(), T.Normalize(MEAN, STD)])
    model.eval(); y_true, y_pred = [], []
    with torch.no_grad():
        for r in df.itertuples():
            x = tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
            p = int(model(x).argmax(1))
            y_true.append(label2idx[r.label]); y_pred.append(p)
    return (accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro", zero_division=0),
            matthews_corrcoef(y_true, y_pred))

CHANCE = 1.0 / NUM_CLASSES
for name, working_version in [("ViTBase16", _timm_working_version), ("SwinTiny", _timm_working_version_swin)]:
    if working_version is None:
        print(f"  [skip] {name}: no working timm version found in the ladder"); continue
    pip_install_version("timm", working_version); purge_module("timm")
    import timm as timm_mod
    try:
        raw = find_and_load_checkpoint(cls_ckpt_candidates(name, 42))
        model = build_backbone_tf(name, NUM_CLASSES, timm_mod)
        model = apply_state(model, raw).to(device).eval()

        val_acc, val_f1, val_mcc = evaluate_classifier_on(val_df, model, IMG_SIZE_CLS_TF)
        tf_val_rows.append({"seed": 42, "model": name, "val_accuracy": val_acc,
                             "val_f1_macro": val_f1, "val_mcc": val_mcc, "timm_version": working_version})
        print(f"  seed42 {name} (timm=={working_version}): val_acc={val_acc:.4f} f1_macro={val_f1:.4f} mcc={val_mcc:.4f}")

        tf = T.Compose([T.Resize((IMG_SIZE_CLS_TF, IMG_SIZE_CLS_TF)), T.ToTensor(), T.Normalize(MEAN, STD)])
        y_true=[]; pred_orig=[]; pred_flip=[]; pred_tta=[]
        with torch.no_grad():
            for r in test_df.itertuples():
                x = tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
                lg_o = model(x); lg_f = model(torch.flip(x, dims=[3]))
                pred_orig.append(int(lg_o.argmax(1))); pred_flip.append(int(lg_f.argmax(1)))
                pred_tta.append(int((lg_o + lg_f).argmax(1)))
                y_true.append(label2idx[r.label])
        y_true = np.array(y_true); po = np.array(pred_orig); pf = np.array(pred_flip); pt = np.array(pred_tta)
        acc_o = (po==y_true).mean(); acc_f = (pf==y_true).mean(); acc_t = (pt==y_true).mean()
        agree = (po==pf).mean()
        corrected = int(((po!=y_true) & (pt==y_true)).sum())
        harmed    = int(((po==y_true) & (pt!=y_true)).sum())
        unchanged = int(len(y_true) - corrected - harmed)
        if acc_o < 2*CHANCE:
            print(f"  *** WARNING: {name} single-pass accuracy ({acc_o:.4f}) is still near chance "
                  f"even with timm=={working_version} -- the ladder's candidate was not the right era. ***")
        tf_tta_rows.append({"model": name, "n_test": len(y_true), "timm_version": working_version,
                             "acc_original_view": float(acc_o), "acc_flipped_view": float(acc_f),
                             "acc_tta_averaged": float(acc_t), "view_agreement_rate": float(agree),
                             "n_corrected_by_tta": corrected, "n_harmed_by_tta": harmed,
                             "n_unchanged_by_tta": unchanged, "tta_minus_original_delta": float(acc_t-acc_o)})
        print(f"  {name}: orig={acc_o:.4f} flip={acc_f:.4f} tta={acc_t:.4f} agree={agree:.4f} "
              f"corrected={corrected} harmed={harmed} unchanged={unchanged}")
    except Exception as e:
        print(f"  [SKIP] {name}: failed even with timm=={working_version} -- {e}")
    finally:
        try: del model
        except NameError: pass
        torch.cuda.empty_cache()

pd.DataFrame(tf_val_rows).to_csv(OUTDIR/"vit_swin_seed42_val_reranking.csv", index=False)
pd.DataFrame(tf_tta_rows).to_csv(OUTDIR/"vit_swin_seed42_tta_breakdown.csv", index=False)
print(f"\nWorking timm version -- ViTBase16: {_timm_working_version}  SwinTiny: {_timm_working_version_swin}")
print("Saved: vit_swin_seed42_val_reranking.csv, vit_swin_seed42_tta_breakdown.csv")

    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/ViTBase16_best.pt
  -- trying timm==1.0.9 --
  >>> SUCCESS: timm==1.0.9 loads this checkpoint cleanly. {'missing': 0, 'unexpected': 0}
    [find_and_load] loaded /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints/SwinTiny_best.pt
  seed42 ViTBase16 (timm==1.0.9): val_acc=0.0922 f1_macro=0.0569 mcc=0.0838
  ViTBase16: orig=0.0833 flip=0.0852 tta=0.0884 agree=0.5490 corrected=27 harmed=19 unchanged=1515
  seed42 SwinTiny (timm==1.0.9): val_acc=0.0095 f1_macro=0.0011 mcc=0.0055
  *** WARNING: SwinTiny single-pass accuracy (0.0122) is still near chance even with timm==1.0.9 -- the ladder's candidate was not the right era. ***
  SwinTiny: orig=0.0122 flip=0.0109 tta=0.0122 agree=0.9110 corrected=0 harmed=0 unchanged=1561

Working timm version -- ViTBase16: 1.0.9  SwinTiny: 1.0.9

## Wrap-up -- pinned-version summary + zip all outputs for download

In [8]:
print("="*70)
print("PINNED VERSIONS THAT ACTUALLY WORKED -- record these in the response letter /")
print("Methods reproducibility note if this run succeeded:")
print(f"  transformers (SegFormer_b2, all 3 seeds): {_sf_working_version}")
print(f"  timm (ViTBase16, seed 42 only):           {_timm_working_version}")
print(f"  timm (SwinTiny, seed 42 only):            {_timm_working_version_swin}")
print("="*70)

import zipfile as _zf
zip_path = Path("/kaggle/working/plantseg_r2_notebookC_results.zip")
with _zf.ZipFile(zip_path, "w", _zf.ZIP_DEFLATED) as zf:
    for p in sorted(OUTDIR.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
print(f"\nArchive: {zip_path.name}  ({zip_path.stat().st_size/1024/1024:.2f} MB)")
print("Files included:")
for p in sorted(OUTDIR.glob("*")):
    print(" ", p.name)
print("\nDownload from the Kaggle Output tab and bring all CSV files + this printed version "
      "summary back for the response letter.")

PINNED VERSIONS THAT ACTUALLY WORKED -- record these in the response letter /
Methods reproducibility note if this run succeeded:
  transformers (SegFormer_b2, all 3 seeds): 4.46.3
  timm (ViTBase16, seed 42 only):           1.0.9
  timm (SwinTiny, seed 42 only):            1.0.9

Archive: plantseg_r2_notebookC_results.zip  (0.01 MB)
Files included:
  segformer_b2_3seed_full_recompute.csv
  segformer_b2_chance_baseline_per_image_seed42.csv
  segformer_b2_val_reranking_3seed.csv
  vit_swin_seed42_tta_breakdown.csv
  vit_swin_seed42_val_reranking.csv

Download from the Kaggle Output tab and bring all CSV files + this printed version summary back for the response letter.
